# Three-Stage Hybrid DDoS Detection Framework
### Hurst-Pearson Statistical Triage → Entropy-GARCH Filter → Autoencoder-Random Forest Classifier

This notebook implements the pipeline described in *"A Three-Stage Hybrid Framework for Real-Time DDoS Detection Using Statistical Triage, Entropy-GARCH Filtering, and Autoencoder-Random Forest Classification"*.

**Pipeline:**
1. **Stage 1 — Statistical Triage:** Hurst exponent (R/S analysis) + Pearson correlation
2. **Stage 2 — Entropy-GARCH Filter:** Shannon entropy of a categorical field, modeled with a GARCH(1,1) volatility process and a dynamic threshold
3. **Stage 3 — AE-RF Classifier:** Autoencoder (trained on benign traffic only) → reconstruction error appended as a feature → Random Forest final verdict
4. **Feedback loop:** Confirmed Stage-3 verdicts periodically recalibrate the Stage-1/Stage-2 thresholds

**Dataset:** CICIDS2017 (Canadian Institute for Cybersecurity Intrusion Detection Dataset). See the *Dataset Download* section below for exact instructions.


## 1. Environment Setup
Install the extra libraries the paper's implementation section names: `arch` (GARCH modeling), `statsmodels`, plus TensorFlow/Keras and scikit-learn (already in Colab).

In [ ]:
!pip install arch --quiet
!pip install kagglehub --quiet
print("Setup complete.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay)
from arch import arch_model
import tensorflow as tf
from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
print("Libraries loaded.")

## 2. Dataset Download — CICIDS2017

The paper evaluates on the **CICIDS dataset** (CICIDS2017), the Canadian Institute for Cybersecurity's benchmark, built with the B-Profile methodology and containing labeled benign + DDoS/attack flow records generated with CICFlowMeter.

**Official source:** https://www.unb.ca/cic/datasets/ids-2017.html
(direct CSV download: `GeneratedLabelledFlows.zip`, specifically the **`Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv`** file for the DDoS-labeled traffic, or use the full `MachineLearningCSV.zip` for all days)

**Easiest route on Colab — Kaggle mirror** (recommended, same data, faster download):
Dataset: `cicids2017` — e.g. https://www.kaggle.com/datasets/cicdataset/cicids2017

Steps:
1. Get a Kaggle API token: Kaggle → Account → *Create New API Token* → downloads `kaggle.json`
2. Upload `kaggle.json` in the cell below when prompted
3. The cell downloads and unzips the CSVs into `/content/cicids2017/`

If you already have the CSV(s) locally, skip this section and just set `DATA_DIR` in the loading cell to point at your files.


In [ ]:
from google.colab import files
import os, zipfile, shutil

# Upload your kaggle.json (Kaggle -> Account -> Create New API Token)
uploaded = files.upload()  # select kaggle.json

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d cicdataset/cicids2017 -p /content/cicids2017 --unzip
print("Download complete. Files:")
for root, dirs, fs in os.walk("/content/cicids2017"):
    for f in fs:
        if f.endswith(".csv"):
            print(os.path.join(root, f))

**Alternative (no Kaggle account):** use `kagglehub` instead of the Kaggle CLI:
```python
import kagglehub
path = kagglehub.dataset_download("cicdataset/cicids2017")
print(path)
```
Or manually upload a CSV via `files.upload()` if you already downloaded one from the official UNB site.


## 3. Load & Preprocess Traffic Flows

Combines the CICIDS2017 CSV(s), cleans missing/infinite values, encodes the label as Benign (0) vs DDoS/Attack (1), and min-max scales the numeric features — matching the paper's preprocessing description in Section 3.1.


In [ ]:
DATA_DIR = "/content/cicids2017"   # change if your CSVs live elsewhere

csv_files = []
for root, dirs, fs in os.walk(DATA_DIR):
    for f in fs:
        if f.lower().endswith(".csv"):
            csv_files.append(os.path.join(root, f))

assert len(csv_files) > 0, "No CSV files found — check DATA_DIR / dataset download step above."
print(f"Found {len(csv_files)} CSV file(s).")

df_list = [pd.read_csv(f, low_memory=False) for f in csv_files]
df = pd.concat(df_list, ignore_index=True)
df.columns = df.columns.str.strip()
print("Combined shape:", df.shape)
df.head()

In [ ]:
# --- Clean ---
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna()

label_col = "Label" if "Label" in df.columns else [c for c in df.columns if c.lower() == "label"][0]
df["binary_label"] = df[label_col].apply(lambda x: 0 if str(x).strip().upper() == "BENIGN" else 1)

print(df["binary_label"].value_counts())
print(f"Benign: {sum(df['binary_label']==0)}  |  DDoS/Attack: {sum(df['binary_label']==1)}")

# --- Feature selection: numeric flow-level features only ---
drop_cols = [label_col, "binary_label"]
feature_cols = [c for c in df.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(df[c])]

X_raw = df[feature_cols].copy()
y = df["binary_label"].values

scaler = MinMaxScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_raw), columns=feature_cols)

print("Feature matrix:", X_scaled.shape)

## 4. Stage 1 — Statistical Triage (Hurst Exponent + Pearson Correlation)

Traffic is grouped into fixed-size windows. For each window:
- **Hurst exponent** `H` is estimated via rescaled-range (R/S) analysis on a per-window time series (here, a packet/byte-rate proxy series), per the paper's formula `H = log(R(n)/S(n)) / log(n)`.
- **Pearson correlation** `r` is computed between pairs of flow-level feature series within the window.

A window is released as **Benign** only if both `H` and `r` fall within empirically calibrated normal ranges; otherwise it is escalated to Stage 2.


In [ ]:
def hurst_exponent(ts, min_lag=2, max_lag=None):
    """Rescaled-range (R/S) Hurst exponent estimate for a 1D series."""
    ts = np.asarray(ts, dtype=float)
    n = len(ts)
    if n < 20:
        return 0.5  # not enough data -> assume normal self-similarity
    if max_lag is None:
        max_lag = n // 2
    lags = range(min_lag, max_lag)
    rs_vals = []
    valid_lags = []
    for lag in lags:
        chunks = n // lag
        if chunks < 1:
            continue
        rs_chunk = []
        for i in range(chunks):
            segment = ts[i*lag:(i+1)*lag]
            mean_adj = segment - segment.mean()
            cum = np.cumsum(mean_adj)
            R = cum.max() - cum.min()
            S = segment.std()
            if S > 0:
                rs_chunk.append(R / S)
        if rs_chunk:
            rs_vals.append(np.mean(rs_chunk))
            valid_lags.append(lag)
    if len(valid_lags) < 2:
        return 0.5
    log_lags = np.log(valid_lags)
    log_rs = np.log(rs_vals)
    H, _ = np.polyfit(log_lags, log_rs, 1)
    return np.clip(H, 0, 1)


WINDOW_SIZE = 200  # flows per window (tune on validation set)
n_windows = len(X_scaled) // WINDOW_SIZE
print(f"Number of Stage-1/2 windows: {n_windows}")

# Proxy time-series columns used for Hurst + Pearson (adjust to columns present in your CSV)
possible_rate_cols = [c for c in feature_cols if "Bytes/s" in c or "Packets/s" in c]
hurst_col = possible_rate_cols[0] if possible_rate_cols else feature_cols[0]
corr_pair = (feature_cols[0], feature_cols[1])

stage1_records = []
for w in range(n_windows):
    start, end = w * WINDOW_SIZE, (w + 1) * WINDOW_SIZE
    window = X_scaled.iloc[start:end]
    ts = window[hurst_col].values
    H = hurst_exponent(ts)
    r, _ = pearsonr(window[corr_pair[0]].values, window[corr_pair[1]].values)
    r = 0.0 if np.isnan(r) else r
    window_label = 1 if y[start:end].sum() > 0 else 0  # window is malicious if it contains any attack flow
    stage1_records.append({"window": w, "H": H, "r": r, "label": window_label,
                            "start": start, "end": end})

stage1_df = pd.DataFrame(stage1_records)

# Empirically calibrate "normal" ranges from windows that are majority-benign
benign_windows = stage1_df[stage1_df["label"] == 0]
H_LOW, H_HIGH = benign_windows["H"].quantile([0.05, 0.95])
R_THRESH = benign_windows["r"].abs().quantile(0.95)

stage1_df["stage1_benign"] = ((stage1_df["H"] >= H_LOW) & (stage1_df["H"] <= H_HIGH)
                               & (stage1_df["r"].abs() <= R_THRESH))
print(f"Stage 1 calibrated ranges -> H in [{H_LOW:.3f}, {H_HIGH:.3f}], |r| <= {R_THRESH:.3f}")
print(f"Stage 1 released as benign: {stage1_df['stage1_benign'].sum()} / {len(stage1_df)} windows")
stage1_df.head()

## 5. Stage 2 — Entropy-Based GARCH Filter

For every window that is **not** released by Stage 1, Shannon entropy `H(X)` of a categorical attribute (destination port, as a proxy for source-IP/destination-port concentration) is computed, then modeled as a **GARCH(1,1)** process to obtain a **dynamic** threshold `λt = μ − k·σt`, rather than a fixed cut-off — matching the paper's Section 3.2.


In [ ]:
def shannon_entropy(series):
    vc = series.value_counts(normalize=True)
    return -np.sum(vc * np.log2(vc + 1e-12))

# Pick a categorical-ish column for entropy (destination port is standard in CICIDS-style datasets)
dport_candidates = [c for c in df.columns if "Dst Port" in c or "Destination Port" in c]
dport_col = dport_candidates[0] if dport_candidates else feature_cols[0]

escalated = stage1_df[~stage1_df["stage1_benign"]].copy()
entropy_series = []
for _, row in escalated.iterrows():
    seg = df[dport_col].iloc[row["start"]:row["end"]]
    entropy_series.append(shannon_entropy(seg))
escalated["entropy"] = entropy_series

# Model the entropy series with GARCH(1,1); scale up for numerical stability (arch expects %-scale returns)
if len(escalated) > 10:
    entropy_returns = escalated["entropy"].diff().dropna() * 100
    garch_model = arch_model(entropy_returns, vol="Garch", p=1, q=1, dist="normal")
    garch_fit = garch_model.fit(disp="off")
    cond_vol = garch_fit.conditional_volatility / 100
    mu = escalated["entropy"].mean()
    k = 1.5  # sensitivity constant
    # align lengths (diff() drops first row)
    cond_vol_full = np.concatenate([[cond_vol.iloc[0]], cond_vol.values])
    lambda_t = mu - k * cond_vol_full[:len(escalated)]
else:
    mu = escalated["entropy"].mean() if len(escalated) else 0
    lambda_t = np.full(len(escalated), mu)

escalated["lambda_t"] = lambda_t
escalated["stage2_benign"] = escalated["entropy"] >= escalated["lambda_t"]

print(f"Stage 2 received: {len(escalated)} windows")
print(f"Stage 2 released as benign: {escalated['stage2_benign'].sum()} / {len(escalated)}")
escalated.head()

## 6. Stage 3 — Autoencoder–Random Forest (AE-RF) Classifier

Only the flows inside windows that are **still suspicious after Stage 2** are escalated here. The autoencoder is trained **only on benign flows** to reconstruct normal traffic; its reconstruction error is appended to the raw feature vector `x' = [x, e]` and fed to a Random Forest for the final Benign/DDoS verdict.


In [ ]:
stage3_windows = escalated[~escalated["stage2_benign"]]
stage3_flow_idx = []
for _, row in stage3_windows.iterrows():
    stage3_flow_idx.extend(range(int(row["start"]), int(row["end"])))

print(f"Flows escalated to Stage 3 (AE-RF): {len(stage3_flow_idx)} / {len(X_scaled)} "
      f"({100*len(stage3_flow_idx)/len(X_scaled):.2f}% of total traffic)")

X_stage3 = X_scaled.iloc[stage3_flow_idx].reset_index(drop=True)
y_stage3 = y[stage3_flow_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X_stage3, y_stage3, test_size=0.3, random_state=42, stratify=y_stage3)

# Autoencoder trained only on benign training flows
X_train_benign = X_train[y_train == 0]

input_dim = X_train.shape[1]
encoding_dim = max(4, input_dim // 4)

autoencoder = models.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(encoding_dim * 2, activation="relu"),
    layers.Dense(encoding_dim, activation="relu"),
    layers.Dense(encoding_dim * 2, activation="relu"),
    layers.Dense(input_dim, activation="sigmoid"),
])
autoencoder.compile(optimizer="adam", loss="mse")

history = autoencoder.fit(
    X_train_benign, X_train_benign,
    epochs=20, batch_size=256, shuffle=True,
    validation_split=0.1, verbose=1
)

In [ ]:
def reconstruction_error(model, X):
    recon = model.predict(X, verbose=0)
    return np.mean(np.square(X.values - recon), axis=1)

train_err = reconstruction_error(autoencoder, X_train)
test_err = reconstruction_error(autoencoder, X_test)

X_train_aug = X_train.copy()
X_train_aug["recon_error"] = train_err
X_test_aug = X_test.copy()
X_test_aug["recon_error"] = test_err

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train_aug, y_train)

y_pred = rf.predict(X_test_aug)
y_proba = rf.predict_proba(X_test_aug)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel() if cm.size == 4 else (0,0,0,0)
fpr_final = fp / (fp + tn) if (fp + tn) > 0 else 0.0

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"FPR:       {fpr_final:.4f}")

## 7. Feedback Loop (Stage 3 → Stage 1/2 Recalibration)

A lightweight illustration of the closed feedback loop: confirmed verdicts are used to widen/tighten the Stage-1 Hurst/Pearson ranges and re-estimate the Stage-2 GARCH baseline mean, so the early stages track gradual traffic drift without full retraining.


In [ ]:
# Confirmed benign flows from Stage 3 feed back into recalibrating Stage 1's thresholds
confirmed_benign_mask = (y_pred == 0)
if confirmed_benign_mask.sum() > 5:
    # Recompute empirical H/r ranges including this newly confirmed benign evidence (illustrative)
    updated_H_low = min(H_LOW, stage1_df.loc[stage1_df["label"]==0, "H"].quantile(0.03))
    updated_H_high = max(H_HIGH, stage1_df.loc[stage1_df["label"]==0, "H"].quantile(0.97))
    print(f"Recalibrated Stage-1 H range: [{updated_H_low:.3f}, {updated_H_high:.3f}] "
          f"(was [{H_LOW:.3f}, {H_HIGH:.3f}])")
else:
    print("Not enough confirmed-benign Stage-3 samples yet to recalibrate thresholds.")

## 8. Result Charts

Reproducing the four visualizations referenced in the paper (Graphs 1–4), plus two supporting diagnostic plots.


### Graph 1 — Stage-wise Traffic Volume Reduction

In [ ]:
stage_counts = {
    "Total Traffic": len(X_scaled),
    "After Stage 1\n(Hurst+Pearson)": len(escalated) * WINDOW_SIZE if len(escalated) else 0,
    "After Stage 2\n(Entropy-GARCH)": len(stage3_windows) * WINDOW_SIZE if len(stage3_windows) else 0,
    "Reaches Stage 3\n(AE-RF)": len(stage3_flow_idx),
}

plt.figure(figsize=(8, 5))
bars = plt.bar(stage_counts.keys(), stage_counts.values(),
                color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
for b in bars:
    plt.text(b.get_x() + b.get_width()/2, b.get_height(), f"{int(b.get_height()):,}",
              ha="center", va="bottom", fontsize=9)
plt.ylabel("Number of Flows")
plt.title("Graph 1: Stage-wise Traffic Volume Reduction")
plt.tight_layout()
plt.show()

### Graph 2 — Performance Comparison vs. Single-Stage Baselines

In [ ]:
# Baseline 1: standalone Random Forest on ALL flows (no triage)
rf_baseline = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y)
rf_baseline.fit(Xb_train, yb_train)
yb_pred = rf_baseline.predict(Xb_test)

baseline_metrics = {
    "Accuracy": accuracy_score(yb_test, yb_pred),
    "Precision": precision_score(yb_test, yb_pred, zero_division=0),
    "Recall": recall_score(yb_test, yb_pred, zero_division=0),
    "F1-score": f1_score(yb_test, yb_pred, zero_division=0),
}
proposed_metrics = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1}

metrics_labels = list(proposed_metrics.keys())
x = np.arange(len(metrics_labels))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, [baseline_metrics[m] for m in metrics_labels], width, label="Standalone RF (baseline)")
plt.bar(x + width/2, [proposed_metrics[m] for m in metrics_labels], width, label="Proposed 3-Stage AE-RF")
plt.xticks(x, metrics_labels)
plt.ylim(0, 1.05)
plt.ylabel("Score")
plt.title("Graph 2: Performance Comparison — Proposed Framework vs. Baseline")
plt.legend()
plt.tight_layout()
plt.show()

### Graph 3 — ROC Curve (Stage 3 AE-RF Classifier)

In [ ]:
fpr_vals, tpr_vals, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr_vals, tpr_vals)

plt.figure(figsize=(6, 6))
plt.plot(fpr_vals, tpr_vals, color="#C44E52", lw=2, label=f"AE-RF (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Graph 3: ROC Curve of the Final Stage-3 Classifier")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Graph 4 — Confusion Matrix (Stage 3 Test Set)

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Benign", "DDoS"])
disp.plot(ax=ax, cmap="Blues", colorbar=False)
plt.title("Graph 4: Confusion Matrix — Stage-3 Test Set")
plt.tight_layout()
plt.show()

### Supporting Plot — Autoencoder Training Loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Reconstruction MSE Loss")
plt.title("Autoencoder Training Curve (benign-only)")
plt.legend()
plt.tight_layout()
plt.show()

### Supporting Plot — Reconstruction Error Distribution (Benign vs. DDoS)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(test_err[y_test == 0], color="#55A868", label="Benign", kde=True, stat="density", alpha=0.5)
sns.histplot(test_err[y_test == 1], color="#C44E52", label="DDoS", kde=True, stat="density", alpha=0.5)
plt.xlabel("Autoencoder Reconstruction Error")
plt.ylabel("Density")
plt.title("Reconstruction Error Distribution by Class (Stage 3 Test Set)")
plt.legend()
plt.tight_layout()
plt.show()

## 9. Summary Table

In [ ]:
summary_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "False Positive Rate",
               "% Traffic reaching Stage 3", "ROC AUC"],
    "Proposed Framework": [acc, prec, rec, f1, fpr_final,
                            len(stage3_flow_idx)/len(X_scaled), roc_auc],
})
summary_df["Proposed Framework"] = summary_df["Proposed Framework"].round(4)
summary_df